In [1]:
# --- MÓDULO DE INGESTÃO: VALOR ADICIONADO BRUTO (VAB) DA INDÚSTRIA ---

import pandas as pd

# Defino o ponto de entrada para os dados do VAB Industrial.
# Esta variável é fundamental para isolar o choque econômico no setor secundário, 
# onde a atividade de extração mineral (Nióbio) está inserida.
caminho_arquivo = r'C:\Users\fabio\TCC\5_VAB_Industria.csv'

try:
    # Realizo a ingestão do dataset de composição setorial.
    # skiprows=1: Tratamento de cabeçalhos institucionais do IBGE/SIDRA.
    # encoding='utf-8-sig': Normalização de caracteres para compatibilidade Windows/Unix.
    df_vab = pd.read_csv(
        caminho_arquivo,
        skiprows=1,
        sep=';',
        encoding='utf-8-sig'
    )
    
    # DIAGNÓSTICO ESTRUTURAL:
    # Verificação da carga para garantir que as séries temporais foram mapeadas corretamente.
    print("--- Visualização do Dataset de Composição Industrial (VAB) ---")
    print(df_vab.head())
    
    print("\n--- Inventário de Séries Temporais Identificadas ---")
    print(df_vab.columns.tolist())

except Exception as e:
    print(f"ERRO DE INGESTÃO: Falha crítica ao carregar a matriz de VAB Industrial: {e}")

--- TABELA CARREGADA ---
  Sigla   Código     Município             1996             1999  \
0    AC  1200013    Acrelândia    792,117122335  2956,7743642211   
1    AC  1200054  Assis Brasil     59,381787402  1132,7171567307   
2    AC  1200104     Brasiléia   983,6287800845   6642,951806159   
3    AC  1200138        Bujari    175,450238174  1802,3365361507   
4    AC  1200179      Capixaba  1250,9889391646  1358,0676318769   

              2000             2001             2002             2003  \
0  4008,5602926495  6622,8914136456  5468,7222071293  4481,1769499115   
1  1216,2875206291  1385,3029843565   1532,289357874  1167,8561428726   
2  7291,9821683772  7747,1980359572  8572,7187482113  5842,1447107668   
3  1961,3953492814  2250,9682088011  2289,1537786416  1709,2347188994   
4  1769,4307442713  2029,1572825794  8516,1671156246  9031,9340264076   

               2004  ...              2013              2014  \
0   6493,2277345094  ...  29349,9451456403  12005,0094341953   

In [3]:
# --- ETAPA 2: NORMALIZAÇÃO E TRANSPOSIÇÃO SETORIAL (VAB INDUSTRIAL) ---

# Nesta etapa, realizo a limpeza de colunas residuais e a reestruturação do 
# dataset para o formato 'Long'. Este procedimento é vital para isolar a 
# performance do setor industrial no modelo de Controle Sintético.

print("Iniciando a higienização dos atributos setoriais...")

# 1. TRATAMENTO DE ATRIBUTOS RESIDUAIS
# Removo colunas vazias geradas por delimitadores excedentes na exportação original.
try:
    df_vab_limpo = df_vab.drop(columns=['Unnamed: 27'])
    print("Sucesso: Coluna residual 'Unnamed: 27' removida.")
except KeyError:
    print("Aviso: Estrutura já normalizada; prosseguindo com a limpeza.")
    df_vab_limpo = df_vab.copy()

# 2. REESTRUTURAÇÃO PARA PAINEL (MELT)
# Transponho a matriz para garantir que a variável 'VAB Ind.' esteja 
# alinhada cronologicamente em uma única série vertical.

# 2a. Definição das Chaves de Identificação Regional
dimensoes_id = ['Sigla', 'Código', 'Município']

# 2b. Execução do Reshaping (Wide to Long)
df_vab_final = df_vab_limpo.melt(
    id_vars=dimensoes_id,
    var_name='Ano',
    value_name='VAB Ind.'
)

# 3. VALIDAÇÃO TÉCNICA DO PAINEL SETORIAL
# Inspeciono os extremos da série para confirmar a integridade do empilhamento.
print("\n--- Diagnóstico: Painel de Valor Adicionado (VAB Industrial) ---")
print(df_pib_longo.head()) # Verificando o início da série histórica
print(df_pib_longo.tail()) # Verificando o período mais recente

Coluna 'Unnamed: 27' removida com sucesso.

--- Tabela 'Derretida' (Formato Longo) ---
  Sigla   Código     Município   Ano         VAB Ind.
0    AC  1200013    Acrelândia  1996    792,117122335
1    AC  1200054  Assis Brasil  1996     59,381787402
2    AC  1200104     Brasiléia  1996   983,6287800845
3    AC  1200138        Bujari  1996    175,450238174
4    AC  1200179      Capixaba  1996  1250,9889391646
       Sigla   Código       Município   Ano          VAB Ind.
134323    TO  1721208  Tocantinópolis  2021  17745,6791007111
134324    TO  1721257        Tupirama  2021  1162,35015022414
134325    TO  1721307      Tupiratins  2021  434,597586288624
134326    TO  1722081    Wanderlândia  2021  9350,15000361024
134327    TO  1722107         Xambioá  2021  107477,243468858


In [4]:
# --- ETAPA 3: SERIALIZAÇÃO E PERSISTÊNCIA DO VAB INDUSTRIAL (OUTPUT) ---

# Finalizo o tratamento da variável de Valor Adicionado Bruto da Indústria. 
# Este dataset é o insumo principal para decompor o crescimento econômico municipal
# e isolar o efeito do setor extrativo na série histórica de Araxá (MG).

# 1. DEFINIÇÃO DO DIRETÓRIO DE SAÍDA
# Armazeno o dataset consolidado utilizando nomenclatura técnica para 
# facilitar a automação no script de integração mestre (Master Merge).
caminho_saida_csv = r'C:\Users\fabio\TCC\5_VAB_IND_FINAL.csv'

# 2. EXPORTAÇÃO E CONFORMIDADE DE DADOS
# O parâmetro index=False assegura que a estrutura do CSV contenha apenas
# as colunas de identificação e a métrica econômica, otimizando o merge no Stata.
df_vab_final.to_csv(
    caminho_saida_csv,
    index=False
)

print(f"--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---")
print(f"Dataset de VAB Industrial Consolidado: {caminho_saida_csv}")

--- SUCESSO! ---
Seu arquivo de população consolidado foi salvo em:
C:\Users\fabio\TCC\5_VAB_IND_FINAL.csv
